Titanic Passenger Analytics — Data Analyst Project
Libraries: NumPy, Pandas, Matplotlib, Seaborn

Dataset: `Titanic-Dataset.csv` (891 rows × 12 columns). Run all cells in order.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
df = pd.read_csv("Titanic-Dataset.csv")
pd.set_option("display.max_columns", None)
print("Shape:", df.shape)
print(df.head(10)); print(df.tail(5)); print(df.dtypes.to_frame("Data Type")); print(df.describe())

A. Data Understanding & NumPy

In [ ]:
age=df.Age.dropna().to_numpy(); fare=df.Fare.dropna().to_numpy()
print("Age:",np.mean(age),np.median(age),np.min(age),np.max(age),np.std(age))
print("Fare:",np.mean(fare),np.median(fare),np.min(fare),np.max(fare),np.var(fare),np.std(fare))
print("Age > 50:",np.sum(age>50))
print("Fare > average:",np.sum(fare>np.mean(fare)))
print("above median fare:",np.mean(fare>np.median(fare))*100)
print("Age range:",np.max(age)-np.min(age))
labels=np.select([age<=12,(age>=13)&(age<=18),(age>=19)&(age<=59),age>=60],["Child","Teenager","Adult","Senior"])
print(pd.Series(labels).value_counts().to_frame("Count").assign(Percentage=lambda x:x.Count/len(age)*100))

## B. Data Cleaning & Demographics

In [ ]:
print(pd.DataFrame({"Missing Count":df.isna().sum(),"Missing %":df.isna().mean()*100}))
print("Duplicate rows:",df.duplicated().sum(),"Unique PassengerId:",df.PassengerId.nunique())
clean=df.copy()
clean["Age"]=clean.Age.fillna(clean.groupby(["Pclass","Sex"]).Age.transform("median"))
clean["Embarked"]=clean.Embarked.fillna(clean.Embarked.mode()[0])
clean["CabinDeck"]=clean.Cabin.astype("string").str[0]
print("Missing Age after imputation:",clean.Age.isna().sum())
print(clean.groupby("Sex").agg(Passengers=("PassengerId","count"),Avg_Age=("Age","mean"),Avg_Fare=("Fare","mean")))
print(clean.groupby("Sex").Survived.agg(Passengers="count",Survivors="sum",Survival_Rate="mean"))
cs=clean.groupby("Pclass").agg(Passengers=("PassengerId","count"),Survivors=("Survived","sum"),Avg_Age=("Age","mean"),Avg_Fare=("Fare","mean"),Survival_Rate=("Survived","mean"))
cs["Non-Survivors"]=cs.Passengers-cs.Survivors
print(cs)

## C. Advanced Pandas Analysis

In [ ]:
clean["FamilySize"]=clean.SibSp+clean.Parch+1
clean["FamilyCategory"]=np.select([clean.FamilySize.eq(1),clean.FamilySize.between(2,4),clean.FamilySize.gt(4)],["Alone","Small Family","Large Family"],default="Unknown")
print(clean.groupby("FamilyCategory").agg(Passengers=("PassengerId","count"),Survivors=("Survived","sum"),Survival_Rate=("Survived","mean")))
print(df.nlargest(10,"Fare")[["PassengerId","Name","Pclass","Fare","Survived"]])
print(clean.groupby("Pclass").Fare.agg(["mean","min","max","std"]))
print(clean.loc[clean.Fare>clean.Fare.mean(),["PassengerId","Name","Pclass","Fare"]])
clean["AgeGroup"]=pd.cut(clean.Age,[-np.inf,12,18,59,np.inf],labels=["Child","Teenager","Adult","Senior"])
print(clean.groupby("AgeGroup",observed=False).agg(Passengers=("PassengerId","count"),Survivors=("Survived","sum"),Survival_Rate=("Survived","mean"),Avg_Fare=("Fare","mean")))
print(clean.groupby(["Sex","Pclass"]).agg(Passenger_Count=("PassengerId","count"),Avg_Age=("Age","mean"),Avg_Fare=("Fare","mean"),Survival_Rate=("Survived","mean")))

## D. Matplotlib Visualizations

In [ ]:
df.Pclass.value_counts().sort_index().plot.bar(); plt.title("Passenger Count by Class"); plt.xlabel("Class"); plt.ylabel("Passengers"); plt.show()
df.Age.dropna().plot.hist(bins=20); plt.title("Age Distribution"); plt.xlabel("Age"); plt.show()
df.Fare.plot.hist(bins=30); plt.title("Fare Distribution"); plt.xlabel("Fare"); plt.show()
counts=df.Survived.value_counts().sort_index(); ax=counts.plot.bar(); ax.set_xticklabels(["Did not survive","Survived"],rotation=0)
for i,v in enumerate(counts): ax.text(i,v+5,str(v),ha="center")
plt.title("Survival Counts"); plt.show()
plt.scatter(df.Age,df.Fare,alpha=.5); plt.title("Age vs Fare"); plt.xlabel("Age"); plt.ylabel("Fare"); plt.show()

## E. Seaborn Visualizations & Correlation

In [ ]:
sns.countplot(data=df,x="Sex",hue="Survived"); plt.title("Survival by Gender"); plt.show()
sns.boxplot(data=clean,x="Sex",y="Age"); plt.title("Age by Gender"); plt.show()
sns.boxplot(data=df,x="Pclass",y="Fare"); plt.title("Fare by Class"); plt.show()
sns.heatmap(df[["Survived","Pclass","Age","SibSp","Parch","Fare"]].corr(),annot=True,fmt=".2f",center=0)
plt.title("Correlation Heatmap"); plt.show()

## Interpretation
Fare is right-skewed, so its median is more representative than its mean. Female passengers had a much higher survival rate. First Class had the highest class survival rate and Third Class the lowest. Correlation is association, not causation. Age group analysis uses imputed ages; NumPy age statistics use observed ages.